# SONYC Label Mapping

This notebook converts SONYC-UST annotations into the five governance classes
used throughout this research.

Target classes:

- Traffic
- Construction
- Entertainment
- Worship
- Ambience

The output will be a clean dataframe that can be used for transfer learning.

In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path

# Project paths

PROJECT = Path("/content/drive/MyDrive/UrbanNoiseProject")

SONYC = PROJECT / "datasets" / "SONYC-UST"

OUTPUT = PROJECT / "datasets" / "processed"

OUTPUT.mkdir(exist_ok=True)

In [8]:
import pandas as pd

# Read annotations

annotations = pd.read_csv(
    SONYC / "annotations.csv"
)

print(annotations.shape)

annotations.head()

(12461, 70)


,split,sensor_id,audio_filename,annotator_id,1-1_small-sounding-engine_presence,1-2_medium-sounding-engine_presence,1-3_large-sounding-engine_presence,1-X_engine-of-uncertain-size_presence,2-1_rock-drill_presence,2-2_jackhammer_presence,...,7-X_other-unknown-human-voice_proximity,8-1_dog-barking-whining_proximity,1_engine_presence,2_machinery-impact_presence,3_non-machinery-impact_presence,4_powered-saw_presence,5_alert-signal_presence,6_music_presence,7_human-voice_presence,8_dog_presence
0,validate,0,00_000066.wav,95,1.0,1.0,1.0,1.0,1.0,1.0,...,far,far,1,1,1,1,1,1,1,1
1,validate,0,00_000066.wav,108,0.0,0.0,1.0,0.0,0.0,0.0,...,-1,-1,1,0,0,0,0,0,0,0
2,validate,0,00_000066.wav,127,1.0,0.0,0.0,0.0,0.0,0.0,...,-1,-1,1,0,0,0,0,0,0,0
3,validate,0,00_000118.wav,45,0.0,0.0,0.0,0.0,0.0,0.0,...,-1,-1,0,0,0,0,0,0,1,0
4,validate,0,00_000118.wav,58,0.0,1.0,0.0,0.0,0.0,0.0,...,-1,-1,1,0,0,0,0,0,1,0


In [9]:
# Keep only clip-level labels

columns = [
    "split",
    "audio_filename",
    "1_engine_presence",
    "2_machinery-impact_presence",
    "3_non-machinery-impact_presence",
    "4_powered-saw_presence",
    "5_alert-signal_presence",
    "6_music_presence",
    "7_human-voice_presence",
    "8_dog_presence"
]

sonyc = annotations[columns].copy()

sonyc.head()

,split,audio_filename,1_engine_presence,2_machinery-impact_presence,3_non-machinery-impact_presence,4_powered-saw_presence,5_alert-signal_presence,6_music_presence,7_human-voice_presence,8_dog_presence
0,validate,00_000066.wav,1,1,1,1,1,1,1,1
1,validate,00_000066.wav,1,0,0,0,0,0,0,0
2,validate,00_000066.wav,1,0,0,0,0,0,0,0
3,validate,00_000118.wav,0,0,0,0,0,0,1,0
4,validate,00_000118.wav,1,0,0,0,0,0,1,0


In [10]:
# Convert SONYC labels to Kigali governance classes

def assign_class(row):

    if row["2_machinery-impact_presence"] == 1:
        return "Construction"

    if row["1_engine_presence"] == 1:
        return "Traffic"

    if row["5_alert-signal_presence"] == 1:
        return "Traffic"

    if row["6_music_presence"] == 1:
        return "Entertainment"

    if row["7_human-voice_presence"] == 1:
        return "Worship"

    return "Ambience"

In [11]:
# Create final class column

sonyc["class"] = sonyc.apply(
    assign_class,
    axis=1
)

sonyc.head()

,split,audio_filename,1_engine_presence,2_machinery-impact_presence,3_non-machinery-impact_presence,4_powered-saw_presence,5_alert-signal_presence,6_music_presence,7_human-voice_presence,8_dog_presence,class
0,validate,00_000066.wav,1,1,1,1,1,1,1,1,Construction
1,validate,00_000066.wav,1,0,0,0,0,0,0,0,Traffic
2,validate,00_000066.wav,1,0,0,0,0,0,0,0,Traffic
3,validate,00_000118.wav,0,0,0,0,0,0,1,0,Worship
4,validate,00_000118.wav,1,0,0,0,0,0,1,0,Traffic


In [12]:
# Remove duplicate annotations

sonyc = sonyc.drop_duplicates(
    subset=["audio_filename"]
)

print(len(sonyc))

3068


In [13]:
# Class distribution

sonyc["class"].value_counts()

,count
class,
Traffic,1085
Ambience,970
Worship,472
Construction,463
Entertainment,78


In [14]:
# Build complete audio paths

def build_path(row):

    return str(
        SONYC /
        row["split"] /
        row["audio_filename"]
    )

sonyc["filepath"] = sonyc.apply(
    build_path,
    axis=1
)

sonyc.head()

/tmp/ipykernel_24490/1546306605.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sonyc["filepath"] = sonyc.apply(


,split,audio_filename,1_engine_presence,2_machinery-impact_presence,3_non-machinery-impact_presence,4_powered-saw_presence,5_alert-signal_presence,6_music_presence,7_human-voice_presence,8_dog_presence,class,filepath
0,validate,00_000066.wav,1,1,1,1,1,1,1,1,Construction,/content/drive/MyDrive/UrbanNoiseProject/datas...
3,validate,00_000118.wav,0,0,0,0,0,0,1,0,Worship,/content/drive/MyDrive/UrbanNoiseProject/datas...
6,validate,00_000275.wav,1,0,1,0,0,0,0,0,Traffic,/content/drive/MyDrive/UrbanNoiseProject/datas...
9,validate,00_000277.wav,0,0,0,0,1,0,0,0,Traffic,/content/drive/MyDrive/UrbanNoiseProject/datas...
12,validate,00_000357.wav,1,0,0,0,0,0,0,0,Traffic,/content/drive/MyDrive/UrbanNoiseProject/datas...


In [15]:
# Verify files

import os

missing = 0

for f in sonyc["filepath"]:

    if not os.path.exists(f):

        missing += 1

print("Missing:", missing)

Missing: 0


In [16]:
# Save processed labels

outfile = OUTPUT / "sonyc_labels.csv"

sonyc.to_csv(
    outfile,
    index=False
)

print(outfile)

/content/drive/MyDrive/UrbanNoiseProject/datasets/processed/sonyc_labels.csv


## Summary

Completed:

- Read SONYC annotations
- Converted labels to governance classes
- Linked annotations with audio files
- Removed duplicate clip annotations
- Saved processed labels

Next notebook:

11_sonyc_transfer_learning.ipynb